# Python Pakete importieren
The NWHD Connector use features of the [PyDEEN Package](https://pypi.org/project/pydeen/). For Data Science activities the pandas library is used to return Dataframe objects.


In [ ]:
# import libs 
import pandas as pd
from pynwhd import NWHDConnector


from pydeen.types import Factory
import matplotlib.pyplot as plt
import seaborn as sns
import calendar 
import panel as pn
import numpy as np


pn.extension('tabulator')


import hvplot.pandas


# Den Connector konfigurieren



In [ ]:
# configure backend and parameters
sid       = "S4D"
desc      = "BA Entwicklung"
client    = "100"
url       = "http://s4d.17.ucc.md"
api_path  = "/nwhd_rest/v1" 
source    = "00505699C4E91EDDA6DDC77FA530E9ED"

date_from = "20230301"
date_to   = "20230430"
max_rows  = 100000

# Das Connector Objekt wird über einen Skript Dialog erstellt - dies ist nur einmal nötig

Beim ersten Start müssen Sie die Authentifizierungsdaten über das Menü eingeben. Sie können sie speichern und für die nächsten Aufrufe wiederverwenden. Die Informationen werden im aktuellen Pfad sicher verschlüsselt.

Es wird eine Liste aller verwendbaren Quellsysteme und der später benötigten Quell-GUID angezeigt. Mit dem Befehl set_source (GUID prüfen) wird eine Liste der verfügbaren numerischen Wertschlüssel angezeigt.

In [ ]:
# Konnektor im Skripting-Modus verwenden und konfigurieren
connector = NWHDConnector(sid, url, client, desc, api_path)
connector.set_date_interval(date_from, date_to)
connector.set_max_rows(max_rows)


#Quellsystem auswählen
connector.set_source(source)

# Fertige Werte holen und bereitstellen

SHMM:SharedMemory:FreeBytes                        =       9200 values
SHMM:SharedMemory:TotalBytes                       =       9200 values
SHMM:SharedMemory:UsedPercent

## Data Frame erstellen


### Erstellen des Standard DataFrames, der dann für alle weiteren Auswertungen herangezogen wird


In [ ]:
# get data as joined dataframe via multiple value columns


set = [
    "WP:All:Runtime"
    ,"UserMemory:SessionMemory:Max"
    ,"UserMemory:SessionMemory:Avg"
    ,"UserMemory:Session:Count"
]

df_UserMem = connector.get_df_numeric_multiple(set)
#print(df_list)   , df_list
#if df_UserMem:  
df_UserMem.plot(figsize=(15, 7))
df_UserMem.dtypes
df_UserMem.index



#else: 
#    print("No joined Data")
    

### Umbenennung von Variablen zur besseren Handhabung

In [ ]:
# List of variable with original name and wanted name

rename_cols = [
    ('WP:All:Runtime', 'Runtime'), 
    ('UserMemory:SessionMemory:Max', 'SessionMemoryMax'), 
    ('UserMemory:SessionMemory:Avg', 'SessionMemoryAvg'), 
    ('UserMemory:Session:Count' , 'SessionCount')
    ]

# Iteration über die Liste und Umbenennung der Spalten
for old_name, new_name in rename_cols:
    df_UserMem.rename(columns={old_name: new_name}, inplace=True)
    
    
df_UserMem.fillna(0, inplace=True)
df_UserMem

In [ ]:
cols = list(df_UserMem.columns)
col1 = cols[0]
col2 = cols[1]
col3 = cols[2]
col4 = cols[3]


df_UM_resampled = df_UserMem.resample('H')

df_base = df_UM_resampled.agg({col1: 'mean', 
                                   col2: 'mean', 
                                   col3: 'mean',
                                   col4: 'max'})
pd.options.display.float_format = '{:.0f}'.format
df_base[[col2 , col3]] /= 1024
df_base

In [ ]:
# get prepared timeseries data for single values
df_UMSum = connector.get_df_numeric("WP:All:Runtime")
df_UMSum.plot(figsize=(15, 7))

df_UMMax = connector.get_df_numeric("UserMemory:SessionMemory:Max")
df_UMMax.plot(figsize=(15, 7))

df_UMAvg = connector.get_df_numeric("UserMemory:SessionMemory:Avg")
df_UMAvg.plot(figsize=(15, 7))

df_UMCnt = connector.get_df_numeric("UserMemory:Session:Count")
df_UMCnt.plot(figsize=(15, 7))

# merge it
# join dataframes via index  // UserMemory:Session:Count // UserMemory:SessionMemory:AVG / MAX / SUM
df_UM = pd.concat([df_UMSum, df_UMMax, df_UMAvg, df_UMCnt ], axis=1)
df_UM
df_UM.plot(figsize=(15, 7))


df_UM.plot.scatter(x='UserMemory:SessionMemory:Max',y='UserMemory:Session:Count',figsize=(15, 7))

### First impression of available dataset

In [ ]:
cols = list(df_base.columns)
x_col = cols[0]
y_col = cols[1]
value_col = cols[2]

cols_subset = cols[1:3] #[:3]
cols_subset

In [ ]:


df_UM_mean = df_UserMem.resample('H').mean()
pd.options.display.float_format = '{:.0f}'.format
##df_UM_mean[['UserMemory:SessionMemory:Sum', 'UserMemory:SessionMemory:Max', 'UserMemory:SessionMemory:Avg']] =df_UM_mean[['UserMemory:SessionMemory:Sum', 'UserMemory:SessionMemory:Max', 'UserMemory:SessionMemory:Avg']].div(1024)
df_UM_mean[cols_subset] = df_UM_mean[cols_subset].div(1024)

df_UM_mean

In [ ]:
cols = list(df_base.columns)
cols_subset = cols[-1:] #[:3]
cols_subset


# Nur die Summe der Anzahl der User Sessions plotten
df_base[cols_subset].plot(figsize=(15, 7))

# Grafik anzeigen
plt.show()

In [ ]:
cols = list(df_base.columns)
#cols_subset = cols[:2] #[:3]
cols_subset = [cols[0], cols[3]]
cols_subset


# Nur gezielte Spalten plotten
pd.options.display.float_format = '{:.0f}'.format
df_base[cols_subset].plot(figsize=(20, 7))
# Grafik anzeigen
plt.show()

In [ ]:
cols = list(df_base.columns)
x_col = cols[2]
y_col = cols[1]
value_col = cols[2]

cols_subset = cols[1:3] #[:1] : nur die erste ##[1:3] Spalte 2 und 3. ## [-1:] nur die letzte ## [:-1] die ersten drei bis auf die letzte
cols

In [ ]:
cols = list(df_base.columns)
col1 = cols[0]
col2 = cols[1]
col3 = cols[2]
col4 = cols[3]
value_col = cols[2]

cols_subset = cols[:-1] #[:3]-Nur die letzte #[:-1] Nur die letzte
cols_subset


sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(18, 9))

f = sns.jointplot(data=df_base, x=col2, y=col1, kind='hex', marginal_ticks=True)

g = sns.displot(data=df_base, x=col4, y=df_base.index, kind='hist', height=6, aspect=1.5, color='skyblue')

a= sns.lineplot(data=df_base[[col1, col4]],linewidth=1.5, ax=ax)

In [ ]:
cols = list(df_base.columns)
col1 = cols[0]
col2 = cols[1]
col3 = cols[2]
col4 = cols[3]
value_col = cols[2]

cols_subset = cols[:-1] #[:3]-Nur die letzte #[:-1] Nur die letzte
cols_subset



pd.options.display.float_format = '{:.0f}'.format
g = sns.displot(
    data=df_base, 
    x=col2, 
    y=col4, 
    kind='hist', 
    height=6, 
    aspect=1.5, 
    color='skyblue')

g.set(title='Histogram of Average Session Memory Usage', xlabel=col2, ylabel=col4)
plt.show()

## CONFIGURING A NEW DATAFRAME

### Now its time to do some Date work

In [ ]:
#Index erstellen und anpassen
 
df_monthly = df_base.reset_index()
df_monthly = df_monthly.sort_index()


In [ ]:
# Neue Spalten erstellen mit den Datumsinformationen

df_monthly['date'] = pd.to_datetime(df_monthly['TIMESTAMP'])
df_monthly['year'] = df_monthly['date'].dt.year
df_monthly['month'] = df_monthly['date'].dt.month
df_monthly['day'] = df_monthly['date'].dt.day
df_monthly['week'] = df_monthly['date'].dt.isocalendar().week # Iso Woche
df_monthly['weekday'] = df_monthly['date'].dt.weekday
df_monthly['Month_Year'] = pd.to_datetime(df_monthly['TIMESTAMP'].dt.to_period('M').astype(str)).dt.strftime('%b %Y')


# Setting Index for the next Function
df_monthly = df_monthly.set_index('TIMESTAMP')

# Create a new column for the weekday
df_monthly['Wochentag'] = df_monthly.index.day_name()
weekdays = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df_monthly['Wochentag'] = pd.Categorical(df_monthly['Wochentag'], categories=weekdays, ordered=True)


# did it work?
df_monthly

In [ ]:
cols = list(df_monthly.columns)
col1 = cols[0]
col2 = cols[1]
col3 = cols[2]
col4 = cols[3]
value_col = cols[3]

cols_subset = cols[:-1] #[:3]-Nur die letzte #[:-1] Nur die letzte
cols_subset

# Heatmaps erstellen

selected_month = ['Mar 2023', 'Apr 2023']

#df_monthly['Month_Year'].unique():

for month_year in selected_month:
    month_df = df_monthly[df_monthly['Month_Year'] == month_year]
    cal_df = month_df.pivot_table(values=col4 
                                  ,index='week' 
                                  ,columns='Wochentag' 
                                  ,aggfunc='max' 
                                  #,fill_value=0
                                 )
    
    cal_df = cal_df.reindex(index=cal_df.index[::1]) # Kalenderwochen umdrehen
    weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    #cal_df = cal_df[weekday_order]


# Heatmap plotten
    fig, ax = plt.subplots(figsize=(15, 8))
    ax.set_title(col4 + ' ' + month_year)
    sns.heatmap(cal_df, cmap='YlGnBu', annot=True, fmt='.0f', annot_kws={"size": 10}, cbar=True, ax=ax, vmin=15, vmax=400, cbar_kws={'label': 'Maximale Anzahl der Sessions'})
    ax.set_xlabel('Wochentage')
    ax.set_ylabel('Kalenderwochen')
    plt.show()  


### Time to create a DataFrame and a Heatmap on hourly base


### Erster Test für die Wochen HeatMap

In [ ]:

df_weekly = df_base.reset_index()
#df_UM1 = df_UM.sort_index()


# Erstelle zusätzliche Spalten für Jahr, Monat, Tag, Stunde und Kalenderwoche
df_weekly['date'] = pd.to_datetime(df_weekly['TIMESTAMP'])
df_weekly['Year'] = df_weekly['TIMESTAMP'].dt.year
df_weekly['Month'] = df_weekly['TIMESTAMP'].dt.month
df_weekly['Day'] = df_weekly['TIMESTAMP'].dt.day
df_weekly['Hour'] = df_weekly['TIMESTAMP'].dt.hour
df_weekly['Week'] = df_weekly['TIMESTAMP'].dt.isocalendar().week

weekdays = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Setting Index for the next Function
df_weekly = df_weekly.set_index('TIMESTAMP')

# Create a new column for the weekday
df_weekly['Wochentag'] = df_weekly.index.day_name()
df_weekly['Wochentag'] = pd.Categorical(df_weekly['Wochentag'], categories=weekdays, ordered=True)


## definiere die Variablen

cols = list(df_weekly.columns)
col1 = cols[0]
col2 = cols[1]
col3 = cols[2]
col4 = cols[3]
value_col = cols[2]

cols_subset = cols[3] #[:3]-Nur die letzte #[:-1] Nur die letzte
cols_subset

# Gruppiere die Daten nach Jahr, Kalenderwoche, Tag und Stunde und berechne die Summe pro Gruppe
dft_grouped = df_weekly.groupby(['Year', 'Month', 'Week', 'Day', 'Hour'])[cols_subset].max().reset_index()



# Erstelle eine Heatmap für jede Kalenderwoche
for week in dft_grouped['Week'].unique():
    # Filtere die Daten für die aktuelle Kalenderwoche
    df_week = dft_grouped[dft_grouped['Week'] == week]

    # Erstelle einen neuen DataFrame, um die Daten für die Heatmap zu formatieren
    df_heatmap = pd.DataFrame(index=range(24), columns=weekdays)

    
    # Fülle den DataFrame mit den gruppierten Daten
for index, row in df_week.iterrows():
    # Erstelle ein datetime-Objekt mit dem Jahr, dem Monat, dem Tag und der Stunde
    dt = pd.Timestamp(year=row['Year'], month=1, day=1) + pd.Timedelta(weeks=row['Week']-1, days=row['Day']-1, hours=row['Hour'])
    # Verwende den Wochentagsnamen als Index für die Zeile
    df_heatmap.loc[row['Hour'], dt.day_name()] = row[cols_subset]
df_heatmap.fillna(0, inplace=True)
    
fig, ax = plt.subplots(figsize=(15, 8))    
sns.heatmap(df_heatmap, annot=True, fmt='.0f', cmap='YlGnBu',annot_kws={"size": 8}, ax=ax)
 

## Now the values have to be split in several HeatMaps 

In [ ]:
df_weekly2 = df_base.reset_index()
#df_UM2.fillna(0, inplace=True)


# Erstelle zusätzliche Spalten für Jahr, Monat, Tag, Stunde und Kalenderwoche
df_weekly2['date'] = pd.to_datetime(df_weekly2['TIMESTAMP'])
df_weekly2['Year'] = df_weekly2['TIMESTAMP'].dt.year
df_weekly2['Month'] = df_weekly2['TIMESTAMP'].dt.month
df_weekly2['Day'] = df_weekly2['TIMESTAMP'].dt.day
df_weekly2['Hour'] = df_weekly2['TIMESTAMP'].dt.hour
df_weekly2['Week'] = df_weekly2['TIMESTAMP'].dt.isocalendar().week

weekdays = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']


# Setting Index for the next Function
df_weekly2 = df_weekly2.set_index('TIMESTAMP')

# Create a new column for the weekday
df_weekly2['Wochentag'] = df_weekly2.index.day_name()
df_weekly2['Wochentag'] = pd.Categorical(df_weekly2['Wochentag'], categories=weekdays, ordered=True)


#Variablen erstellen
cols = list(df_weekly2.columns)
col1 = cols[0]
col2 = cols[1]
col3 = cols[2]
col4 = cols[3]
value_col = cols[2]

cols_subset = cols[3] #[:3]-Nur die letzte #[:-1] Nur die letzte
cols_subset


# Liste mit den Kalenderwochen erstellen, für die eine Heatmap erstellt werden soll
selected_weeks = [ 10, 11, 12, 13, 14, 15, 16]

for week in selected_weeks:
    
    # Subset des DataFrames für die aktuelle Kalenderwoche erstellen
    df_week = df_weekly2[df_weekly2["Week"] == week]
    
    # Pivot-Tabelle erstellen, um Stunden pro Tag als Zeilen und Tage als Spalten anzuzeigen
    df_heatmap = df_week.pivot_table( index='Hour', columns='Wochentag', values=cols_subset)
    df_heatmap = df_heatmap.reindex(index=df_heatmap.index[::-1]) # Stundenanzeige umdrehen
    #df_heatmap.fillna(0, inplace=True)
   
    # Heatmap erstellen
    plt.figure(figsize=(12, 9))
    sns.heatmap(df_heatmap, cmap="YlGnBu", annot=True, fmt=".0f",annot_kws={"size": 8}, vmin=15,vmax=85,  cbar_kws={'label': 'Maximale Anzahl der Sessions'}) 
    plt.title(f"Heatmap {week}. Kalenderwoche")
    plt.xlabel("Tag")
    plt.ylabel("Stunde")
    
    ax.set_xticklabels(df_week['date'].dt.date.unique(), rotation=90)
    plt.show()



## Jetzt noch etwas Statistik

In [ ]:
import statsmodels.api as sm

cols = list(df_UserMem.columns)
col1 = cols[0]
col2 = cols[1]
col3 = cols[2]
col4 = cols[3]
value_col = cols[2]

df_base.fillna(0, inplace=True)
  
# Aufteilen in unabhängige und abhängige Variablen
X = df_UserMem[col4]  # Unabhängige Variable (Anzahl der Sessions)
y = df_UserMem[col2]   # Abhängige Variable (Maximale Speicherauslastung)

# Hinzufügen einer Konstanten zur unabhängigen Variable (wichtig für Regression)
X = sm.add_constant(X)

# Lineare Regression durchführen
model = sm.OLS(y, X).fit()

# Ausgabe der Regressionsergebnisse
print(model.summary())

R-squared: Der Bestimmtheitsgrad (R-squared) beträgt 0,041. Dies bedeutet, dass nur 4,1% der Variabilität in der abhängigen Variablen durch die unabhängige Variable erklärt werden kann.
F-Test: Der F-Test wird verwendet, um zu testen, ob es einen statistisch signifikanten Zusammenhang zwischen den Variablen gibt. Hier ist das F-Verhältnis 696, was sehr hoch ist, und die zugehörige Wahrscheinlichkeit (Prob (F-statistic)) ist sehr niedrig (3.29e-150), was darauf hinweist, dass es eine signifikante Beziehung gibt.

In [ ]:
import seaborn as sns

# Streudiagramm mit Regressionslinie erstellen
sns.lmplot(x=col4, y=col2, data=df_UserMem)

# Achsenbeschriftungen hinzufügen
plt.xlabel('Anzahl der Sessions')
plt.ylabel('Maximale Speicherauslastung')

## Jetzt schauen wir uns noch die Runtime an

In [ ]:
## Der DataFrame df_weekly2 existiert ja bereits - er ist auch die Basis für die Auswertung der Runtime 


#Variablen erstellen
cols = list(df_weekly2.columns)
col1 = cols[0]
col2 = cols[1]
col3 = cols[2]
col4 = cols[3]
value_col = cols[0]

cols_subset = cols[0] #Hier ändere ich das Subset
cols_subset


# Liste mit den Kalenderwochen erstellen, für die eine Heatmap erstellt werden soll
selected_weeks = [ 10, 11, 12, 13, 14, 15, 16]

for week in selected_weeks:
    
    # Subset des DataFrames für die aktuelle Kalenderwoche erstellen
    df_week = df_weekly2[df_weekly2["Week"] == week]
    
    # Pivot-Tabelle erstellen, um Stunden pro Tag als Zeilen und Tage als Spalten anzuzeigen
    df_heatmap = df_week.pivot_table( index='Hour', columns='Wochentag', values=cols_subset)
    df_heatmap = df_heatmap.reindex(index=df_heatmap.index[::-1]) # Stundenanzeige umdrehen
    #df_heatmap.fillna(0, inplace=True)
  
   
    # Heatmap erstellen
    plt.figure(figsize=(12, 9))
    sns.heatmap(df_heatmap, cmap="YlGnBu", annot=True, fmt=".0f",annot_kws={"size": 8}, vmin=100, vmax=350, cbar_kws={'label': 'Durchschnittliche Runtime'}) 
    plt.title(f"Heatmap {week}. Kalenderwoche")
    plt.xlabel("Tag")
    plt.ylabel("Stunde")
    
    ax.set_xticklabels(df_week['date'].dt.date.unique(), rotation=90)
    plt.show()



In [ ]:

import statsmodels.api as sm

cols = list(df_UserMem.columns)
col1 = cols[0] #Runtime
col2 = cols[1] #SessionMemoryMax
col3 = cols[2] #SessionMemoryAvg
col4 = cols[3] #SessionCount


#df_base.fillna(0, inplace=True)
  
# Aufteilen in unabhängige und abhängige Variablen
X = df_UserMem[col2]  # Unabhängige Variable (Maximale Speicherauslastung)
y = df_UserMem[col1]   # Abhängige (Runtime)

# Hinzufügen einer Konstanten zur unabhängigen Variable (wichtig für Regression)
X = sm.add_constant(X)

# Lineare Regression durchführen
model = sm.OLS(y, X).fit()

# Ausgabe der Regressionsergebnisse
print(model.summary())

In [ ]:
# Streudiagramm mit Regressionslinie erstellen
sns.lmplot(x=col2, y=col1, data=df_UserMem)

# Achsenbeschriftungen hinzufügen
plt.xlabel('Maximale Speicherauslastung')
plt.ylabel('Runtime')